In [ ]:
import pandas as pd
import re
from collections import Counter
import json

# Define the 17 refined rejection reasons with keywords for mapping
refined_reasons = {
    "Only Format Change": {
        "keywords": ["format", "formating", "code formatting", "code formating", "clean-up or preference",
                     "spacing", "indentation", "whitespace", "line break", "capitalize", "capitalization",
                     "bold", "italic", "bullet", "list", "code element", "formatting only", "formating only"],
        "patterns": [r"format", r"formating", r"code\s*format", r"clean-up", r"capitalize"]
    },
    "Undesired Spelling/Grammar Fixed": {
        "keywords": ["spelling", "grammar", "grammatical", "typo", "gramar", "grammer",
                     "fixed spelling", "fixed grammar", "uk vs us", "spelling/grammar", "grammar fixed"],
        "patterns": [r"spelling", r"grammar", r"gramar", r"typo", r"uk vs us"]
    },
    "Undesired Text Modification": {
        "keywords": ["text modification", "rewording", "sentence structure", "tense", "voice",
                     "unnecessary text", "unnecessary change", "unnecessary addition",
                     "unnecessary removal", "text change", "unnecessary_phrases", "unnecessary phrase",
                     "gratitude", "thanks", "cheers", "signature", "unnecessary text modification"],
        "patterns": [r"text\s*modification", r"unnecessary\s*(?:text|change|addition|removal)",
                     r"rewording", r"unnecessary_phrases", r"gratitude"]
    },
    "Undesired Code Modification": {
        "keywords": ["code modification", "code change", "refactoring", "variable renaming",
                     "unnecessary code", "code edit", "api change", "command option",
                     "unnecessary code modification", "code modified"],
        "patterns": [r"code\s*modification", r"unnecessary\s*code", r"code\s*change", r"refactor"]
    },
    "Spam link, Keywords": {
        "keywords": ["spam", "spam link", "spam keywords", "spam/vandalism", "promotional",
                     "advertising", "redirect", "spam words", "spam keyword", "spam link"],
        "patterns": [r"spam", r"promotional", r"advertising"]
    },
    "Meaningless Words": {
        "keywords": ["meaningless", "garbled", "nonsensical", "gibberish", "no meaningful",
                     "meaningless word", "garbled text", "garbled post", "grambled",
                     "meaningless words", "no meaningful word", "meaningless word", "garbled"],
        "patterns": [r"meaningless", r"garbled", r"nonsensical", r"gibberish", r"grambled"]
    },
    "Personal Promotion": {
        "keywords": ["personal promotion", "self-promotion", "promote", "personal website",
                     "product promotion", "service promotion", "personal opinion", "promotion"],
        "patterns": [r"personal\s*promotion", r"self-promotion", r"promotion"]
    },
    "Deface Post": {
        "keywords": ["deface", "vandalism", "vandalize", "corrupt", "damage", "deface post"],
        "patterns": [r"deface", r"vandal"]
    },
    "Rude or Abusive Words": {
        "keywords": ["rude", "abusive", "offensive", "profanity", "insult", "abusive word",
                     "rude or abusive", "abusive language", "rude or abusive word"],
        "patterns": [r"rude", r"abusive", r"offensive", r"profanity"]
    },
    "Attempt to Reply": {
        "keywords": ["attempt to reply", "try to reply", "reply within", "answer within",
                     "tried to reply", "reply in post", "attempt reply"],
        "patterns": [r"attempt\s*to\s*reply", r"try\s*to\s*reply", r"reply\s*within"]
    },
    "Comment on Post": {
        "keywords": ["comment on post", "added comment", "comment on question", "discussion",
                     "conversational", "add comment", "comment added", "added a comment"],
        "patterns": [r"comment\s*on\s*(?:post|question)", r"added\s*comment", r"discussion"]
    },
    "Status Update": {
        "keywords": ["status update", "update about", "newer version", "version update",
                     "status add", "status addd", "deprecation", "status", "added an update"],
        "patterns": [r"status\s*update", r"newer\s*version", r"version\s*update", r"status\s*add"]
    },
    "Invalid Correction": {
        "keywords": ["invalid correction", "incorrect correction", "wrong fix", "incorrect change",
                     "incorrect information", "potentially incorrect", "invalid change",
                     "incorrect correction", "invalid edit", "probably invalid"],
        "patterns": [r"invalid\s*(?:correction|change)", r"incorrect\s*(?:correction|change|information)",
                     r"wrong\s*fixed", r"potentially\s*incorrect"]
    },
    "Incorrect Syntax": {
        "keywords": ["incorrect syntax", "syntax error", "invalid syntax", "syntactic error",
                     "incorrect syntax", "syntax", "syntactic", "invalid syntax"],
        "patterns": [r"incorrect\s*syntax", r"syntax\s*error", r"invalid\s*syntax"]
    },
    "Alternate Suggestion": {
        "keywords": ["alternate suggestion", "alternative solution", "different solution",
                     "alternate solution", "different approach", "alternative suggestion",
                     "suggested a different solution"],
        "patterns": [r"alternate?\s*(?:suggestion|solution)", r"different\s*solution", r"alternative\s*solution"]
    },
    "Removed Content's Purpose": {
        "keywords": ["removed content purpose", "remove purpose", "changed purpose",
                     "remove contents purpose", "alter intent", "essential removal",
                     "remove clarification", "removed reference", "over-simplification",
                     "removed content purpose", "remove contents purpose"],
        "patterns": [r"remove.*purpose", r"content['\s]*purpose", r"essential\s*removal", r"over-simplification"]
    },
    "Question on Post": {
        "keywords": ["question on post", "follow up question", "follow-up question",
                     "question on question", "follow up questions", "asking question",
                     "added question", "followup question", "follow-up questions"],
        "patterns": [r"question\s*on\s*(?:post|question)", r"follow[-_\s]*up\s*question", r"asking\s*question"]
    }
}

def map_reason_to_category(reason_text):
    """Map a single unstructured reason to one of the 17 refined categories"""
    if pd.isna(reason_text) or reason_text is None:
        return "Other/Unmapped"

    text = str(reason_text).lower().strip()

    # Skip empty or header rows
    if not text or text == "manual reason" or text == "manual reason":
        return "Other/Unmapped"

    # Check each category
    for category, info in refined_reasons.items():
        for keyword in info["keywords"]:
            if keyword.lower() in text:
                return category

        for pattern in info["patterns"]:
            if re.search(pattern, text, re.IGNORECASE):
                return category

    # Special cases for common patterns
    if "general minor edit" in text:
        if "no change" in text or "no meaningful" in text:
            return "Only Format Change"
        elif "format" in text or "grammar" in text:
            return "Only Format Change"
        else:
            return "Only Format Change"

    if "unnecessary" in text:
        if "code" in text:
            return "Undesired Code Modification"
        elif "addition" in text or "removal" in text or "modification" in text:
            return "Undesired Text Modification"
        else:
            return "Undesired Text Modification"

    if "minor edit" in text:
        if "title" in text or "tag" in text:
            return "Question on Post"  # title/tag edits are minor but map appropriately
        else:
            return "Only Format Change"

    if "no change" in text or "no meaningful change" in text:
        return "Only Format Change"

    if "clarification" in text:
        if "added" in text or "add" in text:
            return "Undesired Text Modification"
        elif "remove" in text:
            return "Removed Content's Purpose"

    if "reference" in text:
        if "removed" in text:
            return "Removed Content's Purpose"
        elif "added" in text:
            return "Spam link, Keywords"

    return "Other/Unmapped"

def process_csv_file(csv_path, column_name="Manual Reason"):
    """Process CSV file and map rejection reasons"""

    # Read CSV file
    df = pd.read_csv(csv_path)

    # Check if column exists
    if column_name not in df.columns:
        print(f"Error: Column '{column_name}' not found in CSV file.")
        print(f"Available columns: {list(df.columns)}")
        return None

    # Map each reason to category
    df['Mapped_Category'] = df[column_name].apply(map_reason_to_category)

    # Calculate statistics
    category_counts = df['Mapped_Category'].value_counts()
    total = len(df)
    category_percentages = (category_counts / total) * 100

    # Create summary DataFrame
    summary_df = pd.DataFrame({
        'Count': category_counts,
        'Percentage': category_percentages
    }).sort_values('Count', ascending=False)

    return df, summary_df

def print_results(summary_df, total):
    """Print formatted results"""
    print("=" * 80)
    print("MAPPING RESULTS FOR 17 REFINED REJECTION REASONS")
    print("=" * 80)
    print(f"\nTotal entries processed: {total}\n")

    print("-" * 80)
    print(f"{'Refined Rejection Reason':<35} {'Count':<10} {'Percentage':<10}")
    print("-" * 80)

    # Print in the order of refined_reasons
    for reason in refined_reasons.keys():
        if reason in summary_df.index:
            count = summary_df.loc[reason, 'Count']
            pct = summary_df.loc[reason, 'Percentage']
            print(f"{reason:<35} {count:<10} {pct:.1f}%")

    # Print Other/Unmapped if exists
    if 'Other/Unmapped' in summary_df.index:
        count = summary_df.loc['Other/Unmapped', 'Count']
        pct = summary_df.loc['Other/Unmapped', 'Percentage']
        print(f"{'Other/Unmapped':<35} {count:<10} {pct:.1f}%")

    print("-" * 80)
    print(f"{'TOTAL':<35} {total:<10} {100.0:.1f}%")

def generate_latex_table(summary_df, total):
    """Generate LaTeX table rows"""
    latex_rows = []

    for reason in refined_reasons.keys():
        if reason in summary_df.index:
            count = summary_df.loc[reason, 'Count']
            pct = (count / total) * 100
            latex_rows.append(f"{reason} & {count} ({pct:.1f}\\%) \\\\")

    # Add Other category if exists
    if 'Other/Unmapped' in summary_df.index:
        count = summary_df.loc['Other/Unmapped', 'Count']
        pct = (count / total) * 100
        if count > 0:
            latex_rows.append(f"Other/Unmapped & {count} ({pct:.1f}\\%) \\\\")

    return latex_rows

def export_mapped_csv(df, output_path="mapped_rejection_reasons.csv"):
    """Export mapped DataFrame to CSV"""
    df.to_csv(output_path, index=False)
    print(f"\nMapped data exported to: {output_path}")

def export_mapping_dictionary(df, column_name="Manual Reason", output_path="rejection_mapping.json"):
    """Export mapping dictionary for future use"""
    mapping_dict = {}
    for original, mapped in zip(df[column_name], df['Mapped_Category']):
        if pd.notna(original) and str(original).strip():
            mapping_dict[str(original)] = mapped

    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(mapping_dict, f, indent=2, ensure_ascii=False)

    print(f"Mapping dictionary exported to: {output_path}")

# Main execution
if __name__ == "__main__":
    # Specify your CSV file path
    CSV_FILE_PATH = "/content/Ultimate Category.csv"  # <-- CHANGE THIS TO YOUR CSV FILE PATH

    # Column name containing the rejection reasons
    REASON_COLUMN = "Manual Reason"  # Change if column name is different

    try:
        # Process the CSV file
        mapped_df, summary_df = process_csv_file(CSV_FILE_PATH, REASON_COLUMN)

        if mapped_df is not None:
            # Print results
            total = len(mapped_df)
            print_results(summary_df, total)

            # Generate LaTeX table rows
            latex_rows = generate_latex_table(summary_df, total)
            print("\n" + "=" * 80)
            print("LATEX TABLE ROWS (copy into your table):")
            print("=" * 80)
            for row in latex_rows:
                print(f"  {row}")

            # Export results
            export_mapped_csv(mapped_df, "mapped_rejection_reasons.csv")
            export_mapping_dictionary(mapped_df, REASON_COLUMN, "rejection_mapping.json")

            # Show sample of mapped data
            print("\n" + "=" * 80)
            print("SAMPLE OF MAPPED DATA (first 10 rows):")
            print("=" * 80)
            sample_df = mapped_df[[REASON_COLUMN, 'Mapped_Category']].head(10)
            for idx, row in sample_df.iterrows():
                print(f"  {row[REASON_COLUMN][:50]:<50} -> {row['Mapped_Category']}")

    except FileNotFoundError:
        print(f"Error: File '{CSV_FILE_PATH}' not found.")
        print("Please update the CSV_FILE_PATH variable with the correct path.")
    except Exception as e:
        print(f"An error occurred: {str(e)}")

MAPPING RESULTS FOR 17 REFINED REJECTION REASONS

Total entries processed: 858

--------------------------------------------------------------------------------
Refined Rejection Reason            Count      Percentage
--------------------------------------------------------------------------------
Only Format Change                  279        32.5%
Undesired Spelling/Grammar Fixed    11         1.3%
Undesired Text Modification         109        12.7%
Undesired Code Modification         30         3.5%
Spam link, Keywords                 22         2.6%
Meaningless Words                   39         4.5%
Personal Promotion                  7          0.8%
Deface Post                         11         1.3%
Rude or Abusive Words               5          0.6%
Attempt to Reply                    12         1.4%
Comment on Post                     19         2.2%
Status Update                       5          0.6%
Invalid Correction                  6          0.7%
Incorrect Syntax      

In [ ]:
import pandas as pd
a=pd.read_csv("/content/mapped_rejection_reasons.csv")
a

,SuggestedEditId,PostId,Sug_Comment,Sug_Text,ph_text,Manual Reason,Ultimate Category Codes,Ultimate Categories,Primary Ultimate Code,Primary Ultimate Category,Mapped_Category
0,NaN,Q-No improvements Whatsoeever,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Other/Unmapped
1,3006846,42584926,should use formatted content,I just installed bootstrap into my rails appli...,I just installed bootstrap into my rails proje...,General Minor Edit - formating,R2,Unnecessary Formatting or Cosmetic Change,R2,Unnecessary Formatting or Cosmetic Change,Only Format Change
2,5481050,75509380,proper error formatting,My application fails at runtime after updating...,My application fails at runtime after updating...,General Minor Edit - formating,R2,Unnecessary Formatting or Cosmetic Change,R2,Unnecessary Formatting or Cosmetic Change,Only Format Change
3,3213233,45359923,Typo in title,NaN,I have two directives in my code where parent ...,minor edit in title/tag,R1,No Meaningful Im-,R1,No Meaningful Im-,Question on Post
4,3556898,48734717,inline code,I'm pulling my hair out trying to center this ...,Im pulling my hair out trying to center this p...,unnecessary code addition,R3,Unnecessary Content Addition or Removal,R3,Unnecessary Content Addition or Removal,Undesired Code Modification
...,...,...,...,...,...,...,...,...,...,...,...
853,1363059,23955801,"$im = imagecreatefrompng (""image/white-round.p...","So, I have a `padding-top` to define some spac...","So, i have a `padding-top` to define some spac...",NaN,NaN,NaN,UNKNOWN,UNKNOWN,Other/Unmapped
854,746696,4836297,"$index = imagecolorclosest ( $im, 255,255,255 );","I am using inner join, left outer join, right ...","I am using inner join, left outer join, right ...",NaN,NaN,NaN,UNKNOWN,UNKNOWN,Other/Unmapped
855,1459829,14517857,NaN,[100.81.195.4][1]Yesterday morning I published...,Yesterday morning I published my app on the go...,NaN,NaN,NaN,UNKNOWN,UNKNOWN,Other/Unmapped
856,142196,8227898,"imagecolorset($im,$index,0,0,255);",I am in trouble please help me out.I want to s...,I am in trouble please help me out.I want to s...,Edit,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,Other/Unmapped
